<a href="https://colab.research.google.com/github/AmandhaPanagoda/AmandhaPanagoda/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_DIR   = '/content/drive/MyDrive/FYP Research/Notebooks/Data'
PROC_DIR   = '/content/drive/MyDrive/FYP Research/Notebooks/Processed'
OUTPUT_DIR = PROC_DIR

In [9]:
# helper functions
def find_file(patterns, data_dir=DATA_DIR):
  if not os.path.isdir(data_dir):
    return None
  for pat in patterns:
    for f in os.listdir(data_dir):
      if pat.lower() in f.lower():
        return os.path.join(data_dir, f)
  return None

In [7]:
# load master_pairs
pairs_path = os.path.join(PROC_DIR, 'master_pairs.parquet')
master_pairs = pd.read_parquet(pairs_path)

print(f'master_pairs: {master_pairs.shape}')
print(master_pairs.dtypes)
print()
display(master_pairs.head(5))

master_pairs: (180514, 9)
ModelID             object
DRUG_ID              int64
DRUG_NAME           object
LN_IC50            float64
AUC                float64
TCGA_DESC           object
SANGER_MODEL_ID     object
COSMIC_ID            int64
CELL_LINE_NAME      object
dtype: object



,ModelID,DRUG_ID,DRUG_NAME,LN_IC50,AUC,TCGA_DESC,SANGER_MODEL_ID,COSMIC_ID,CELL_LINE_NAME
0,ACH-001711,1003,Camptothecin,-1.4639,0.9302,MB,SIDM01132,683667,PFSK-1
1,ACH-000052,1003,Camptothecin,-4.8695,0.6150,UNCLASSIFIED,SIDM00848,684052,A673
2,ACH-000087,1003,Camptothecin,-5.1430,0.5824,UNCLASSIFIED,SIDM01111,684072,SK-ES-1
3,ACH-000644,1003,Camptothecin,-1.2350,0.8673,SKCM,SIDM00909,687448,COLO-829
4,ACH-000905,1003,Camptothecin,-2.6326,0.8341,BLCA,SIDM00807,687452,5637


In [8]:
# define the training cohort
cohort_ids = sorted(master_pairs['ModelID'].unique())
cohort_set = set(cohort_ids)

print(f'training cohort: {len(cohort_ids)} cell lines')
print(f'first 5 ModelIDs: {cohort_ids[:5]}')


training cohort: 713 cell lines
first 5 ModelIDs: ['ACH-000001', 'ACH-000002', 'ACH-000004', 'ACH-000006', 'ACH-000007']


In [10]:
expr_path = find_file(['OmicsExpressionProteinCodingGenesTPMLogp1'])
print(f'loading: {expr_path}')
expr = pd.read_csv(expr_path, index_col=0)
print(f'raw shape: {expr.shape}  (cell lines × genes)')

loading: /content/drive/MyDrive/FYP Research/Notebooks/Data/OmicsExpressionProteinCodingGenesTPMLogp1.csv
raw shape: (1673, 19193)  (cell lines × genes)


In [11]:
expr_ids = set(expr.index.astype(str))
missing_from_expr = cohort_set - expr_ids
print(f'cohort lines missing from expression: {len(missing_from_expr)}')
if missing_from_expr:
  print('  ', list(missing_from_expr)[:10])

expr = expr[expr.index.astype(str).isin(cohort_set)]
expr.index = expr.index.astype(str)
print(f'after cohort filter: {expr.shape}')

cohort lines missing from expression: 0
after cohort filter: (713, 19193)


In [12]:
gene_var = expr.var(axis=0)
low_var_mask = gene_var < 0.01
print(f'genes with variance < 0.01: {low_var_mask.sum():,}')
expr = expr.loc[:, ~low_var_mask]
gene_var = gene_var[~low_var_mask]
print(f'after low-variance filter: {expr.shape}')

genes with variance < 0.01: 984
after low-variance filter: (713, 18209)


In [13]:
top_genes_expr = gene_var.nlargest(2000).index.tolist()  # checkfor several values
expr = expr[top_genes_expr]
print(f'{expr.shape}')

(713, 2000)
